# ЛР6 · Репетиция AI-части Demo Day

Пять кейсов на теге `lab6-ai-demo-v1`. Каждый прошёл три независимых холодных раунда через FastAPI, генератор и SQLite. После прогрева имитируется отказ провайдера. Ячейки читают сохранённый реальный прогон. Команда повторения приведена в `docs/demo-run.md`.

In [1]:
import csv
import os
import json
from pathlib import Path
from IPython.display import Markdown, display

relative = Path("docs/demo/results/lab6-local")
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / relative).is_dir())
os.chdir(root)
folder = root / relative
summary = json.loads((folder / "summary.json").read_text(encoding="utf-8"))
answers = json.loads((folder / "answers.json").read_text(encoding="utf-8"))
with (folder / "requests.csv").open(encoding="utf-8", newline="") as f:
    requests = list(csv.DictReader(f))
with (folder / "failure-checks.csv").open(encoding="utf-8", newline="") as f:
    failures = list(csv.DictReader(f))

def table(headers, body):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines += ["| " + " | ".join(map(str, row)) + " |" for row in body]
    display(Markdown("\n".join(lines)))

display(Markdown(f"Тег `{summary['tag']}` · коммит `{summary['revision']}` · режим `{summary['mode']}`"))
table(["Кейс", "Прогоны", "p50, мс", "p95/max, мс", "Различных наборов тем"],
      [[name, m["requests"], m["p50_ms"], m["max_ms"],
        len({tuple(a["titles"]) for a in answers if a["case_id"] == name})]
       for name, m in summary["per_case"].items()])


Тег `lab6-ai-demo-v1` · коммит `3a6e9eb4190759ac9b0ef9967ae9e6b97a8957ab` · режим `local`

| Кейс | Прогоны | p50, мс | p95/max, мс | Различных наборов тем |
| --- | --- | --- | --- | --- |
| history-long | 3 | 21840.614 | 31783.093 | 1 |
| injection-json | 3 | 8450.766 | 9939.832 | 1 |
| databases | 3 | 9112.162 | 9675.951 | 1 |
| devops | 3 | 9076.077 | 9552.774 | 2 |
| edu-biblio | 3 | 8724.77 | 9383.367 | 1 |

In [2]:
from collections import Counter
table(["Проверка", "Повторов", "Статус API", "LLM-вызовов"],
      [[name, len(group), sorted({x["status"] for x in group}),
        sum(int(x["provider_calls"]) for x in group)]
       for name in dict.fromkeys(x["scenario"] for x in failures)
       for group in [[x for x in failures if x["scenario"] == name]]])
assert len(failures) == summary["failure_checks"]
assert all(x["provider_calls"] == "0" for x in failures if x["scenario"] == "warm_cache_503")
display(Markdown("Таймаут имитируется на границе HTTP-клиента. Измеренная задержка отражает быстрый отказ и ретраи."))


| Проверка | Повторов | Статус API | LLM-вызовов |
| --- | --- | --- | --- |
| warm_cache_503 | 15 | ['200'] | 0 |
| cold_503 | 3 | ['503'] | 9 |
| cold_timeout | 3 | ['503'] | 9 |
| cold_401 | 3 | ['503'] | 3 |
| stream_503 | 3 | ['200'] | 9 |

Таймаут имитируется на границе HTTP-клиента. Измеренная задержка отражает быстрый отказ и ретраи.

In [3]:
import sys
sys.path.insert(0, str(root))
from ml.experiments.lab6 import replay

backup = replay(folder, "databases")
assert backup["mode"] == "recorded_demo"
display(Markdown("**Сохранённый ответ PostgreSQL**"))
for title in backup["titles"]:
    display(Markdown(title))


**Сохранённый ответ PostgreSQL**

Разработка системы адаптивной оптимизации запросов в PostgreSQL с использованием механизма динамического планирования и индексов-скриптов для поддержки сложных OLTP-запросов в реальном времени

In [4]:
display(Markdown("**Кейс ограничения**"))
for answer in answers:
    if answer["case_id"] == "edu-biblio":
        display(Markdown(f"Раунд {answer['round']}. " + " / ".join(answer["titles"])))


**Кейс ограничения**

Раунд 1. Разработка интеллектуальной системы автоматизации анализа учебных траекторий студентов с использованием нейросетей для прогнозирования успеваемости и рекомендаций по индивидуализации обучения

Раунд 2. Разработка интеллектуальной системы автоматизации анализа учебных траекторий студентов с использованием нейросетей для прогнозирования успеваемости и рекомендаций по индивидуализации обучения

Раунд 3. Разработка интеллектуальной системы автоматизации анализа учебных траекторий студентов с использованием нейросетей для прогнозирования успеваемости и рекомендаций по индивидуализации обучения

Показанные числа относятся к автономному API AI-ветки. Командный прод, общая презентация и независимая экспертная оценка требуют отдельных подтверждений. Широкая профильность, учёт конкретного focus и смысловые дубли рассматриваются отдельно в документах ЛР5.